In [29]:
import pandas as pd

df = pd.read_csv("../../data/1_derived/3_sf_properties_census_geocoded.csv")
print(f"Shape: {df.shape}")
df.head()

Shape: (9352, 34)


,LeaseCompId,PropertyId,PropertyName,Tenant.Name,TenantIndustry,TenantNAICS,Market,Submarket,Address.DeliveryAddress,Suite,...,longitude,state_fips,county_fips,census_tract,census_block,geoid,tie_matched_addresses,tie_latitudes,tie_longitudes,tie_geoids
0,167100221.0,36156.0,NaN,Millican Jones Interior Planning + Design,Services,NaN,San Francisco,Union Square,445 Bush St,NaN,...,-122.404826,6.0,75.0,11700.0,4006.0,6.075012e+13,NaN,NaN,NaN,NaN
1,162545361.0,36157.0,New Montgomery Tower,Socotra Inc,Information,Software Publishers,San Francisco,South Financial District,33 New Montgomery St,290,...,-122.401551,6.0,75.0,61501.0,3000.0,6.075062e+13,NaN,NaN,NaN,NaN
2,70221222.0,36158.0,115 Sansome St,NaN,NaN,NaN,San Francisco,Financial District,115 Sansome St,NaN,...,-122.400850,6.0,75.0,11700.0,2018.0,6.075012e+13,NaN,NaN,NaN,NaN
3,133318191.0,36159.0,Second Street Square,NaN,NaN,NaN,San Francisco,Rincon/South Beach,501 2nd St,NaN,...,-122.393193,6.0,75.0,61508.0,3001.0,6.075062e+13,NaN,NaN,NaN,NaN
4,70087748.0,36160.0,NaN,NaN,NaN,NaN,San Francisco,Financial District,110 Sutter St,NaN,...,-122.402722,6.0,75.0,11700.0,4003.0,6.075012e+13,NaN,NaN,NaN,NaN


## Missing Value Analysis: Latitude, Longitude, GEOID

In [30]:
geo_cols = ["latitude", "longitude", "geoid"]

# --- Missing counts & percentages ---
missing = df[geo_cols].isnull().sum().rename("missing_count")
missing_pct = (missing / len(df) * 100).rename("missing_pct").round(2)
summary = pd.concat([missing, missing_pct], axis=1)
summary["present_count"] = len(df) - summary["missing_count"]
summary.index.name = "column"
print(f"Total rows: {len(df)}\n")
print(summary.to_string())

Total rows: 9352

           missing_count  missing_pct  present_count
column                                              
latitude             354         3.79           8998
longitude            354         3.79           8998
geoid                354         3.79           8998


## Unmatched Records: `match` ≠ "Match"

In [31]:
# --- Rows where geocoder did NOT return a "Match" ---
cols_to_show = ["PropertyId", "PropertyName", "Address.DeliveryAddress",
                "Address.PostalCode","full_address_std", "match", "matched_address",
                "latitude", "longitude", "geoid"]

no_match_mask = df["match"] != "Match"
no_match_rows = df.loc[no_match_mask, [c for c in cols_to_show if c in df.columns]]

print(f"Total rows without a 'Match': {no_match_mask.sum()} of {len(df)} ({no_match_mask.mean()*100:.1f}%)")
print(f"\nUnique values in 'match' column: {df['match'].unique()}\n")

print(f"Showing first {min(20, len(no_match_rows))} rows:")
no_match_rows

Total rows without a 'Match': 372 of 9352 (4.0%)

Unique values in 'match' column: ['Match' 'No_Match' 'Tie' nan]

Showing first 20 rows:


,PropertyId,PropertyName,Address.DeliveryAddress,Address.PostalCode,full_address_std,match,matched_address,latitude,longitude,geoid
6,36162.0,NaN,710 Market St,94102.0,"710 MARKET ST, SAN FRANCISCO, CA 94102, UNITED...",No_Match,NaN,NaN,NaN,NaN
27,36889.0,Genesis SSF - North Tower,2 Tower Pl,94080.0,"2 TOWER PL, SOUTH SAN FRANCISCO, CA 94080, UNI...",No_Match,NaN,NaN,NaN,NaN
52,47879.0,NaN,1405 Huntington Ave,94080.0,"1405 HUNTINGTON AVE, SOUTH SAN FRANCISCO, CA 9...",No_Match,NaN,NaN,NaN,NaN
77,52103.0,NaN,690 Market St,94104.0,"690 MARKET ST, SAN FRANCISCO, CA 94104, UNITED...",No_Match,NaN,NaN,NaN,NaN
86,53542.0,Pier 29,Pier 29,94121.0,"PIER 29, SAN FRANCISCO, CA 94121, UNITED STATES",No_Match,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
9227,11097600.0,NaN,3300 Marina Blvd,94005.0,"3300 MARINA BLVD, BRISBANE, CA 94005, UNITED S...",No_Match,NaN,NaN,NaN,NaN
9239,11135003.0,NaN,734 El Camino Real,94070.0,"734 EL CAMINO REAL, SAN CARLOS, CA 94070, UNIT...",No_Match,NaN,NaN,NaN,NaN
9306,11462093.0,NaN,1001 Oreilly Ave,94129.0,"1001 OREILLY AVE, SAN FRANCISCO, CA 94129, UNI...",No_Match,NaN,NaN,NaN,NaN
9313,11584338.0,Presidio Officers Club,50 Moraga Ave,94129.0,"50 MORAGA AVE, SAN FRANCISCO, CA 94129, UNITED...",No_Match,NaN,NaN,NaN,NaN


## Tie Records Export

In [32]:
tie_cols = ["full_address_std", "suite", "IsVerified.RawValue", "SpaceUse", 
            "LeaseSource", "LeaseOrigin", "PropertyName", "Tenant.Name", 
            "TenantIndustry", "TenantNAICS"] + [c for c in df.columns if c.startswith("tie_")]

# Filter to only columns that exist in the dataframe
tie_cols = [c for c in tie_cols if c in df.columns]

tie_df = df.loc[df["match"] == "Tie", tie_cols].copy()

out_path = "../../temp/tie_records.csv"
tie_df.to_csv(out_path, index=False)

print(f"Tie rows: {len(tie_df)}")
print(f"Columns: {tie_cols}")
print(f"Saved to: {out_path}")
tie_df

Tie rows: 18
Columns: ['full_address_std', 'IsVerified.RawValue', 'SpaceUse', 'LeaseSource', 'LeaseOrigin', 'PropertyName', 'Tenant.Name', 'TenantIndustry', 'TenantNAICS', 'tie_matched_addresses', 'tie_latitudes', 'tie_longitudes', 'tie_geoids']
Saved to: ../../temp/tie_records.csv


,full_address_std,IsVerified.RawValue,SpaceUse,LeaseSource,LeaseOrigin,PropertyName,Tenant.Name,TenantIndustry,TenantNAICS,tie_matched_addresses,tie_latitudes,tie_longitudes,tie_geoids
972,"1485 BAYSHORE BLVD, SAN FRANCISCO, CA 94124, U...",False,Office,Third Party - Other,CoStar Research,NaN,NaN,NaN,NaN,"1485 BAY SHORE BLVD, SAN FRANCISCO, CA, 94124 ...",37.725535736384 | 37.725535736384,-122.401401989518 | -122.401401989518,060750233002003 | 060750233002003
2801,"3130 LA SELVA DR, SAN MATEO, CA 94403, UNITED ...",False,Office,Third Party - Other,CoStar Research,NaN,NaN,NaN,NaN,"3130 LA SELVA CIR, SAN MATEO, CA, 94403 | 3130...",37.542678745951 | 37.54307237241,-122.284437924233 | -122.285084558557,060816084003008 | 060816084003001
3829,"26 PIER, SAN FRANCISCO, CA 94105, UNITED STATES",False,Office/Retail,Third Party - Other,CoStar Research,NaN,Pilara Family Foundation,Services,Grantmaking and Giving Services,"26 PIER, SAN FRANCISCO, CA, 94111 | 26 PIER, S...",37.795961269966 | 37.740561465372,-122.395085944366 | -122.376715590602,060750105002002 | 060759809001017
4602,"256 EL CAMINO REAL, SAN MATEO, CA 94401, UNITE...",False,Office,Third Party - Other,CoStar Research,NaN,NaN,NaN,NaN,"256 S EL CAMINO REAL, SAN MATEO, CA, 94401 | 2...",37.563454197511 | 37.568959419958,-122.326559076275 | -122.333189329068,060816064005002 | 060816059022004
4794,"100 SKYLINE PLZ, DALY CITY, CA 94015, UNITED S...",False,Office,Third Party - Landlord Rep,CoStar Research,NaN,Pace Learning Center,Health Care and Social Assistance,Child Day Care Services,"100 SKYLINE BLVD, DALY CITY, CA, 94015 | 100 S...",37.673925650836 | 37.691764185401,-122.488596769759 | -122.494884037259,060816015011000 | 060816010004000
5132,"PIER 5 PACIFIC AVE, SAN FRANCISCO, CA 94105, ...",False,Office,Third Party - Landlord Rep,CoStar Research,NaN,Geolo Capital,Finance and Insurance,Portfolio Management and Investment Advice,"5 PACIFIC AVE, SAN FRANCISCO, CA, 94111 | 5 PA...",37.798097684223 | 37.796195392821,-122.397413747842 | -122.412913979,060750105002006 | 060750108002001
6102,"435 BROADWAY BLVD, SAN FRANCISCO, CA 94133, UN...",False,Retail,Third Party - Tenant Rep,CoStar Research,NaN,Broadway Studios,Accommodation and Food Services,Drinking Places (Alcoholic Beverages),"435 BROADWAY ST, SAN FRANCISCO, CA, 94133 | 43...",37.798087228536 | 37.798087228536,-122.404443200708 | -122.404443200708,060750106002006 | 060750106002006
6110,"88 N HILL DR, BRISBANE, CA 94005, UNITED STATES",False,Office,Third Party - Landlord Rep,CoStar Research,NaN,NaN,NaN,NaN,"88 S HILL DR, BRISBANE, CA, 94005 | 88 W HILL ...",37.686556712518 | 37.692860513616,-122.411619079175 | -122.421788578945,060816001002017 | 060816001002013
6237,"313 SAN MATEO DR, SAN MATEO, CA 94401, UNITED ...",False,Retail,Third Party - Other,CoStar Research,NaN,My Day Wedding and Photo Studio,NaN,NaN,"313 S SAN MATEO DR, SAN MATEO, CA, 94401 | 313...",37.564197193327 | 37.571051391926,-122.323970606746 | -122.331329802994,060816063003022 | 060816059021003
6359,"751 SAN BRUNO AVE, SAN BRUNO, CA 94066, UNITED...",False,Retail,Third Party - Landlord Rep,CoStar Research,NaN,NaN,NaN,NaN,"751 SAN BRUNO AVE W, SAN BRUNO, CA, 94066 | 75...",37.628528342383 | 37.630969151044,-122.417215794707 | -122.406114264877,060816040001002 | 060816042001004
